<table width="100%" bgcolor="#0B1F3A" style="background-color:#0B1F3A;background-image:url('https://images.unsplash.com/photo-1576086213369-97a306d36557?auto=format&fit=crop&w=1600&q=80');background-size:cover;border-radius:22px;"><tr><td style="padding:36px 40px;background-color:rgba(11,31,58,.78);">
<p style="letter-spacing:.2em;text-transform:uppercase;font-size:12px;color:#F4C95D;">CZ BIOHUB · 3D+T TRACKING · CODE COMPETITION</p>
<h1 style="font-size:34px;line-height:1.15;color:#F8FAFC;-webkit-text-fill-color:#F8FAFC;text-shadow:0 2px 16px #0B1F3A;">One epoch lost 0.0046 on one FOV</h1>
<p style="font-size:18px;line-height:1.5;color:#F8FAFC;">A reproducible no-hub tracker that still writes <b>submission.csv</b>. Temporal UNet + transformer, ILP, one-frame gap repair.</p>
<p style="font-size:13px;color:#67E8E2;">Den Pugovkin · GPU · Internet off</p>
</td></tr></table>

## TL;DR

- **Submission:** This notebook writes `/kaggle/working/submission.csv` from the attached support-pack checkpoint. Its previously published no-hub parent scored **0.885 public LB**; this rebuilt notebook has no independent LB score yet.
- **Metric:** adjusted edge Jaccard + 0.1 × division Jaccard. Nodes match inside **7 µm**. The host's current division metric requires local directed branches.
- **Detector and graph:** peak threshold **0.970**, 8-view YX TTA, ILP, one-frame gap close, short-track filter. There is **no fake ancestral hub**.
- **Negative result:** `exp002` fine-tuned one epoch on four FOVs and lost **0.004601** versus the original checkpoint on one held-out FOV (0.973699 vs 0.978300). That candidate is excluded from this submission.
- **Reproduce:** Attach the Biohub competition and `pilkwang/biohub-tracking-support-pack-50ep-v1` **v10**, select GPU, disable Internet, then Run All. Kaggle's visible `test/` is an example copy; scoring substitutes hidden volumes.

The single-FOV fine-tune result is diagnostic, not full CV. The pretrained checkpoint's original training-FOV provenance is unknown.


## Contents

1. [The 7 µm contract](#1)
2. [Load a volume](#2)
3. [Detect and link](#3)
4. [Solve the graph with ILP](#4)
5. [Repair the forest](#5)
6. [Write submission.csv](#6)


<h2 id="1" style="color:#0B1F3A;border-bottom:3px solid #00B4A6;padding-bottom:6px;">1. The 7 µm contract</h2>

Question: what does the scorer actually compare, and why can the number exceed 1.0?


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display

INK, PAPER, NAVY = "#1C1917", "#F6F1E7", "#0B1F3A"
TEAL, GOLD, CORAL = "#00B4A6", "#F4C95D", "#E85D4C"
LAYOUT = dict(
    paper_bgcolor=PAPER, plot_bgcolor=PAPER,
    font=dict(color=INK, size=13),
    margin=dict(l=48, r=24, t=56, b=40), height=420,
)
SCALE_UM = np.array([1.625, 0.40625, 0.40625])
MATCH_UM = 7.0


def draw(fig):
    fig.update_layout(**LAYOUT)
    fig.show()


In [ ]:
contract = pd.DataFrame({
    "piece": ["node match", "edge TP", "N_pred term", "division"],
    "rule": [
        "bipartite assignment, max 7 µm",
        "both endpoints matched and GT-linked",
        "adj = J * (1 - 0.1 * (N_pred - N_true) / N_true)",
        "node with >=2 children; lineage covers both daughters",
    ],
    "trap": [
        "far dummy nodes never match",
        "unmatched nodes are not edge FPs",
        "no clamp at 1, so under-count can score above 1",
        "connectivity can matter more than local mitosis",
    ],
})
display(HTML(contract.to_html(index=False)))
print("voxel um", SCALE_UM.tolist(), "match gate", MATCH_UM)


In [ ]:
jaccard = np.linspace(0.70, 0.98, 40)
fig = go.Figure()
for ratio, color in zip([0.7, 1.0, 1.3, 1.8], [TEAL, NAVY, GOLD, CORAL]):
    adj = np.maximum(0, jaccard * (1 - 0.1 * (ratio - 1)))
    fig.add_trace(go.Scatter(
        x=jaccard, y=adj, mode="lines",
        name=f"N_pred / N_true = {ratio}",
        line=dict(color=color, width=3),
    ))
fig.update_layout(
    title="Under-counting nodes can lift adjusted Jaccard above raw J",
    xaxis_title="edge Jaccard J", yaxis_title="adj_edge_jaccard",
)
draw(fig)


**Measured ablation (`exp002`).** The same decoder and FOV gave 0.978300 with the original checkpoint and 0.973699 after one epoch on four training FOVs. Node recall fell from 0.99303 to 0.98490. That FOV contains no division events; it cannot validate division quality. The safe submission choice is the original support-pack checkpoint until broader paired validation says otherwise.

<div style="background:#F6F1E7;border-left:6px solid #F4C95D;padding:14px 18px;border-radius:0 12px 12px 0;margin:12px 0;"><b>Decision.</b> Prefer a high peak threshold and a sparse graph. Do not dump every bright voxel into <code>N_pred</code>. The patched division metric rewards local directed branches. We do not add synthetic ancestors.</div>


<h2 id="2" style="color:#0B1F3A;border-bottom:3px solid #00B4A6;padding-bottom:6px;">2. A volume, not a table</h2>

Question: what is actually on disk, and does one MIP look like sparse cells?


In [ ]:
import os, sys, glob, json, csv, subprocess, importlib, importlib.util

os.environ.setdefault("POLARS_PREFER_PKG", "32")
SLUG = "biohub-cell-tracking-during-development"
PACK_HINTS = [
    Path("/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1"),
    Path("/kaggle/input/biohub-tracking-support-pack-50ep-v1"),
]
DATA_HINTS = [
    Path(f"/kaggle/input/competitions/{SLUG}"),
    Path(f"/kaggle/input/{SLUG}"),
]


def first_dir(paths):
    for path in paths:
        if path.exists():
            return path
    return None


In [ ]:
def find_pack() -> Path:
    direct = first_dir(PACK_HINTS)
    if direct is not None:
        return direct
    root = Path("/kaggle/input")
    if root.exists():
        hits = list(root.glob("**/edge_predictor_best.pth"))
        if hits:
            return hits[0].parents[3]
    raise FileNotFoundError("Attach pilkwang/biohub-tracking-support-pack-50ep-v1")


PACK = find_pack()
DATA = first_dir(DATA_HINTS)
if DATA is None:
    raise FileNotFoundError("Competition data not mounted")
print("pack", PACK)
print("data", DATA)
print("pack children", sorted(p.name for p in PACK.iterdir())[:12])


In [ ]:
WHEELS = PACK / "wheels"
if not WHEELS.exists():
    found = list(Path("/kaggle/input").glob("**/wheels"))
    if not found:
        raise FileNotFoundError("offline wheels directory missing")
    WHEELS = found[0]

NEED = {
    "tracksdata": "tracksdata", "zarr": "zarr", "numcodecs": "numcodecs",
    "geff": "geff", "pyscipopt": "pyscipopt", "ilpy": "ilpy",
    "rustworkx": "rustworkx", "polars": "polars", "imagecodecs": "imagecodecs",
}
missing = [pkg for pkg, mod in NEED.items() if importlib.util.find_spec(mod) is None]
print("wheels", WHEELS)
print("missing", missing or "none")


In [ ]:
PACKAGES = [
    "tracksdata", "zarr==3.2.1", "numcodecs==0.15.1", "donfig==0.8.1.post1",
    "geff==1.2.0.1.1", "geff-spec==1.1.1", "pyscipopt==6.2.1", "ilpy==0.6.0",
    "rustworkx==0.18.0", "polars==1.42.0", "polars-runtime-32==1.42.0",
    "bidict==0.23.1", "imagecodecs==2026.6.26",
]
if missing:
    cmd = [sys.executable, "-m", "pip", "install", "--quiet", "--no-index",
           "--no-deps", "--find-links", str(WHEELS), *PACKAGES]
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.returncode != 0:
        print(result.stdout[-2000:], result.stderr[-2000:])
        raise RuntimeError("offline install failed")
    for root in NEED.values():
        for name in list(sys.modules):
            if name == root or name.startswith(root + "."):
                sys.modules.pop(name, None)
print("offline install ok")


In [ ]:
import zarr, geff, tracksdata as td
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from joblib import Parallel, delayed
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from tqdm.auto import tqdm
import rustworkx as rx

sys.path.append(str(PACK / "repo" / "src"))
from biohub_tracking.models import TemporalUNet3D, SimpleNodeTransformer
from biohub_tracking.io import open_dataset, save_graph

print("zarr", zarr.__version__, "torch", torch.__version__)
print("cuda", torch.cuda.is_available(), "gpus", torch.cuda.device_count())


In [ ]:
image_dir = DATA / 'test'
sample_ids = sorted(p.stem for p in image_dir.glob('*.zarr'))
if not sample_ids:
    raise FileNotFoundError(f'No test volumes in {image_dir}')
print('hidden-test compatible input:', image_dir, 'FOVs:', len(sample_ids))
print(sample_ids)


In [ ]:
probe = image_dir / f"{sample_ids[0]}.zarr"
vol = zarr.open_group(str(probe), mode="r")["0"]
frame = np.asarray(vol[0, :, ::4, ::4], dtype=np.float32)
mip = frame.max(axis=0)
lo, hi = np.percentile(mip, [1, 99.5])
show = np.clip((mip - lo) / (hi - lo + 1e-6), 0, 1)
fig = go.Figure(go.Heatmap(z=show[::-1], colorscale="Turbo", showscale=False))
fig.update_layout(
    title=f"MIP of t=0 · {sample_ids[0]} · shape {tuple(vol.shape)}",
    xaxis_title="x (subsampled)", yaxis_title="y (subsampled)",
    height=520, yaxis=dict(scaleanchor="x"),
)
draw(fig)
print("volume", tuple(vol.shape), "dtype", vol.dtype)


<div style="background:#F6F1E7;border-left:6px solid #00B4A6;padding:14px 18px;border-radius:0 12px 12px 0;margin:12px 0;"><b>Decision.</b> Interactive test is four FOVs copied from train. Hidden scoring swaps a larger set of similar size. Do not tune thresholds on this MIP; it is a sanity picture, not CV.</div>


<h2 id="3" style="color:#0B1F3A;border-bottom:3px solid #00B4A6;padding-bottom:6px;">3. Detect, then link</h2>

Question: how does a 64³ Temporal UNet become a sparse graph?


In [ ]:
SUBSAMPLE = [1, 4, 4]
VOLUME_SHAPE = [64, 64, 64]
TIME_LENGTH = 2
POINT_THRESHOLD = 0.9700
POOL_KERNEL_UM = 3.0
USE_TTA = True
USE_MULTI_GPU = True
ILP_EDGE_WEIGHT = -1.0
ILP_APPEARANCE_WEIGHT = 0.0
ILP_DISAPPEARANCE_WEIGHT = 1.4
ILP_DIVISION_WEIGHT = 1.0
EDGE_STRONG = 0.50
EDGE_MIN = 0.25
EDGE_TOPK = 2
EDGE_MAX_UM = 10.0
print("peak", POINT_THRESHOLD, "tta", USE_TTA, "disappear", ILP_DISAPPEARANCE_WEIGHT)


In [ ]:
class TrackingNet(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.anchor = nn.Parameter(torch.ones(1))
        width = int(config["unet_out_channels"])
        self.unet = TemporalUNet3D(
            in_channels=1, out_channels=width,
            layers=tuple(config["unet_layers"]), gradient_checkpointing=False,
        )
        self.detect_head = nn.Conv3d(width, 1, kernel_size=1)
        self.transformer = SimpleNodeTransformer(
            feat_dim=width + 32, hidden_dim=128, n_heads=4, n_blocks=4, dropout=0,
        )

    def encode(self, image):
        features = self.unet(image[:, :, None])
        logits = [self.detect_head(features[:, 0]), self.detect_head(features[:, 1])]
        return [features[:, 0], features[:, 1]], logits

    def link_nodes(self, sel0, sel1, c0, c1, p0, p1):
        f0 = torch.cat([sel0, p0], dim=-1)
        f1 = torch.cat([sel1, p1], dim=-1)
        return self.transformer(f0, f1, c0, c1)


In [ ]:
def fourier_embed(zyx, t, pos_per_dim=8):
    zyx = zyx.float()
    z, y, x = zyx.unbind(1)
    scale = torch.as_tensor(t, dtype=zyx.dtype, device=zyx.device) / TIME_LENGTH
    axes = [torch.ones_like(z) * scale, z / VOLUME_SHAPE[0],
            y / VOLUME_SHAPE[1], x / VOLUME_SHAPE[2]]

    def embed(values):
        freqs = 2.0 ** torch.arange(pos_per_dim // 2, dtype=values.dtype, device=values.device)
        angles = values[:, None] * freqs[None, :] * torch.pi
        return torch.cat([torch.sin(angles), torch.cos(angles)], 1)

    return torch.cat([embed(axis) for axis in axes], 1)


def odd_kernel(um, voxel_size):
    out = []
    for step in voxel_size:
        k = max(1, round(um / step))
        out.append(k + 1 if k % 2 == 0 else k)
    return tuple(out)


In [ ]:
def peaks_from_prob(prob, threshold, pool_kernel):
    batched = prob.unsqueeze(0)
    pad = tuple(k // 2 for k in pool_kernel)
    pooled = F.max_pool3d(batched, pool_kernel, stride=1, padding=pad)
    peak_idx = torch.nonzero(((batched == pooled) & (batched > threshold))[0, 0])
    if peak_idx.shape[0] == 0:
        return torch.empty((0, 3), dtype=torch.long, device=prob.device)
    return peak_idx


def take_features(feature, zyx):
    _, z_max, y_max, x_max = feature.shape
    z = zyx[:, 0].long().clamp(0, z_max - 1)
    y = zyx[:, 1].long().clamp(0, y_max - 1)
    x = zyx[:, 2].long().clamp(0, x_max - 1)
    return feature[:, z, y, x].permute(1, 0).contiguous()


In [ ]:
def tta_yx8(image):
    dims = (-2, -1)
    views = [
        image,
        image.flip(dims=(-1,)),
        image.flip(dims=(-2,)),
        image.flip(dims=(-2, -1)),
        torch.rot90(image, 1, dims=dims),
        torch.rot90(image, 3, dims=dims),
        image.transpose(-1, -2),
        torch.rot90(image, 1, dims=dims).transpose(-1, -2),
    ]
    return views


def invert_tta_yx8(views):
    dims = (-2, -1)
    return torch.stack([
        views[0], views[1].flip(dims=(-1,)), views[2].flip(dims=(-2,)),
        views[3].flip(dims=(-2, -1)), torch.rot90(views[4], -1, dims=dims),
        torch.rot90(views[5], -3, dims=dims), views[6].transpose(-1, -2),
        torch.rot90(views[7].transpose(-1, -2), -1, dims=dims),
    ])


In [ ]:
def candidate_edges(edge_prob, coord0, coord1, raw_voxel):
    n_src, n_tgt = edge_prob.shape
    if n_src == 0 or n_tgt == 0:
        return []
    pairs = {(int(i), int(j)) for i, j in np.argwhere(edge_prob >= EDGE_STRONG)}
    top_k = min(EDGE_TOPK, n_src)
    for j in range(n_tgt):
        col = edge_prob[:, j]
        idx = np.arange(n_src) if top_k == n_src else np.argpartition(col, -top_k)[-top_k:]
        for i in idx:
            if float(col[i]) >= EDGE_MIN:
                pairs.add((int(i), int(j)))
    out = []
    for i, j in pairs:
        dist = float(np.linalg.norm((coord0[i] - coord1[j]) * raw_voxel))
        if dist <= EDGE_MAX_UM:
            out.append((float(edge_prob[i, j]), i, j, dist))
    out.sort(reverse=True)
    return out


In [ ]:
class TrackBuffer:
    def __init__(self):
        self.nodes, self.edges, self.start = [], [], {}

    def add_pair(self, edge_prob, coord0, coord1, t0, t1, raw_voxel):
        if t0 == 0:
            self.start[t0] = len(self.nodes)
            self.nodes.extend([t0, float(z), float(y), float(x)] for z, y, x in coord0)
        self.start[t1] = len(self.nodes)
        self.nodes.extend([t1, float(z), float(y), float(x)] for z, y, x in coord1)
        s0, s1 = self.start[t0], self.start[t1]
        for prob, i, j, dist in candidate_edges(edge_prob, coord0, coord1, raw_voxel):
            self.edges.append([i + s0, j + s1, prob, dist])


In [ ]:
def load_volume(sample_id):
    path = image_dir / f"{sample_id}.zarr"
    ds = open_dataset(path, normalize=False, load_image=False, require_tracks=False)
    arr = zarr.open_group(str(ds.zarr_path), mode="r")["0"]
    q_low, q_high = float(ds.quantiles["0.001"]), float(ds.quantiles["0.999"])
    dz, dy, dx = SUBSAMPLE
    small = arr[:, ::dz, ::dy, ::dx].astype(np.float32)
    assert small.shape[1:] == tuple(VOLUME_SHAPE)
    small = np.clip((small - q_low) / (q_high - q_low + 1e-6), 0.0, None)
    voxel_size = tuple(s * d for s, d in zip(ds.scale, SUBSAMPLE))
    return small, {"voxel_size": voxel_size}


def load_weights(path, model):
    state = torch.load(path, map_location="cpu", weights_only=True)
    missing, unexpected = model.load_state_dict(state, strict=False)
    print("loaded", path.name, "missing", len(missing), "unexpected", len(unexpected))
    return model


In [ ]:
def encode_pair(model, volume, t):
    pair = torch.from_numpy(volume[t:t + 2]).to(model.anchor.device)
    views = tta_yx8(pair) if USE_TTA else [pair]
    with torch.inference_mode():
        feats, logits = model.encode(torch.stack(views, 0))
        if USE_TTA:
            feats = [invert_tta_yx8(item) for item in feats]
            logits = [invert_tta_yx8(item) for item in logits]
            f0, f1 = feats[0].mean(0), feats[1].mean(0)
        else:
            f0, f1 = feats[0][0], feats[1][0]
        prob = [torch.sigmoid(item.mean(0) if USE_TTA else item[0]) for item in logits]
    return f0, f1, prob


In [ ]:
def predict_volume(model, volume, meta):
    device = model.anchor.device
    voxel = meta["voxel_size"]
    pool = odd_kernel(POOL_KERNEL_UM, voxel)
    subsample = torch.as_tensor([SUBSAMPLE], dtype=torch.float32, device=device)
    raw_voxel = np.asarray(voxel, np.float64) / np.asarray(SUBSAMPLE, np.float64)
    buf, zyx0 = TrackBuffer(), None
    with torch.inference_mode():
        for t in tqdm(range(volume.shape[0] - 1), leave=False):
            f0, f1, prob = encode_pair(model, volume, t)
            if t == 0:
                zyx0 = peaks_from_prob(prob[0], POINT_THRESHOLD, pool)
            zyx1 = peaks_from_prob(prob[1], POINT_THRESHOLD, pool)
            c0, c1 = zyx0 * subsample, zyx1 * subsample
            p0, p1 = fourier_embed(zyx0, 0), fourier_embed(zyx1, 1)
            s0 = take_features(f0, zyx0).unsqueeze(0)
            s1 = take_features(f1, zyx1).unsqueeze(0)
            bsz = len(s0)
            logit = model.link_nodes(
                s0, s1, c0[None].expand(bsz, -1, -1), c1[None].expand(bsz, -1, -1),
                p0[None].expand(bsz, -1, -1), p1[None].expand(bsz, -1, -1),
            )
            eprob = torch.softmax(logit.mean(0), dim=0)
            buf.add_pair(eprob.detach().float().cpu().numpy(),
                         c0.detach().float().cpu().numpy(),
                         c1.detach().float().cpu().numpy(), t, t + 1, raw_voxel)
            zyx0 = zyx1
    return buf.nodes, buf.edges


In [ ]:
def graph_from_buffer(nodes, edges):
    graph = td.graph.InMemoryGraph()
    for key in ("z", "y", "x"):
        graph.add_node_attr_key(key, pl.Float64, -999999.0)
    node_ids = graph.bulk_add_nodes(
        [{"t": int(t), "z": float(z), "y": float(y), "x": float(x)} for t, z, y, x in nodes]
    )
    if edges:
        graph.add_edge_attr_key("edge_prob", pl.Float64, 0.0)
        graph.add_edge_attr_key("edge_dist", pl.Float64, 0.0)
        graph.bulk_add_edges([
            {"source_id": node_ids[i], "target_id": node_ids[j],
             "edge_prob": prob, "edge_dist": dist}
            for i, j, prob, dist in edges
        ])
    return graph


<div style="background:#F6F1E7;border-left:6px solid #00B4A6;padding:14px 18px;border-radius:0 12px 12px 0;margin:12px 0;"><b>Decision.</b> Keep detections sparse (0.97) and edges local (10 µm). The transformer only proposes parents; ILP decides which survive. 8-view YX TTA averages both heatmaps and edge features.</div>


<h2 id="4" style="color:#0B1F3A;border-bottom:3px solid #00B4A6;padding-bottom:6px;">4. ILP on two GPUs</h2>

Question: can we decode a globally consistent forest from those candidates?


In [ ]:
WEIGHT_DIR = PACK / "weights" / "unet_transformer" / "split_0"
CHECKPOINT = WEIGHT_DIR / "edge_predictor_best.pth"
CONFIG_PATH = WEIGHT_DIR / "config.json"
if not CHECKPOINT.is_file() or not CONFIG_PATH.is_file():
    raise FileNotFoundError("Attach support pack v10 with checkpoint and config")
from hashlib import sha256 as checkpoint_sha256
digest = checkpoint_sha256(CHECKPOINT.read_bytes()).hexdigest()
if digest != "12f6881ee3620a831697ca098ff8f48e687a24225f4e048b538deec3562fe771":
    raise ValueError("Support-pack checkpoint differs from the published parent")
print("pinned checkpoint SHA-256:", digest)
PREDICT_DIR = Path("/kaggle/working/graphs")
PREDICT_DIR.mkdir(parents=True, exist_ok=True)
COLUMNS = [
    "id", "dataset", "row_type", "node_id", "t", "z", "y", "x",
    "source_id", "target_id",
]


def solve_ilp(graph):
    if graph.num_edges() == 0:
        return graph
    solver = td.solvers.ILPSolver(
        edge_weight=ILP_EDGE_WEIGHT * td.EdgeAttr("edge_prob"),
        appearance_weight=ILP_APPEARANCE_WEIGHT,
        disappearance_weight=ILP_DISAPPEARANCE_WEIGHT,
        division_weight=ILP_DIVISION_WEIGHT,
        num_threads=1,
    )
    return solver.solve(graph)


In [ ]:
def run_worker(gpu_id, subset):
    torch.cuda.set_device(gpu_id)
    device = torch.device(f"cuda:{gpu_id}")
    with CONFIG_PATH.open() as handle:
        config = json.load(handle)
    model = load_weights(CHECKPOINT, TrackingNet(config)).to(device).eval()
    for sample_id in subset:
        volume, meta = load_volume(sample_id)
        nodes, edges = predict_volume(model, volume, meta)
        graph = solve_ilp(graph_from_buffer(nodes, edges))
        save_graph(graph, str(PREDICT_DIR / f"{sample_id}.geff"))
    del model
    torch.cuda.empty_cache()
    return gpu_id


In [ ]:
if USE_MULTI_GPU and torch.cuda.device_count() >= 2:
    jobs = Parallel(n_jobs=2, backend="loky", verbose=10)([
        delayed(run_worker)(0, sample_ids[0::2]),
        delayed(run_worker)(1, sample_ids[1::2]),
    ])
    print("workers", jobs)
else:
    run_worker(0, sample_ids)
print("graphs", len(list(PREDICT_DIR.glob("*.geff"))))


In [ ]:
def graph_to_rows(sample_id, graph):
    nodes = list(graph.node_attrs().iter_rows(named=True))
    edges = list(graph.edge_attrs().iter_rows(named=True))
    if not nodes:
        raise AssertionError(f"{sample_id}: empty graph")
    seen = {int(row["node_id"]) for row in nodes}
    rows = []
    for row in sorted(nodes, key=lambda item: int(item["node_id"])):
        rows.append({
            "id": -1, "dataset": sample_id, "row_type": "node",
            "node_id": int(row["node_id"]), "t": int(row["t"]),
            "z": max(0, int(round(float(row["z"])))),
            "y": max(0, int(round(float(row["y"])))),
            "x": max(0, int(round(float(row["x"])))),
            "source_id": -1, "target_id": -1,
        })
    for row in edges:
        src, tgt = int(row["source_id"]), int(row["target_id"])
        if src not in seen or tgt not in seen:
            raise AssertionError(f"{sample_id}: dangling {src}->{tgt}")
        rows.append({
            "id": -1, "dataset": sample_id, "row_type": "edge",
            "node_id": -1, "t": -1, "z": -1, "y": -1, "x": -1,
            "source_id": src, "target_id": tgt,
        })
    return rows


In [ ]:
raw_rows, n_nodes, n_edges = [], 0, 0
for sample_id in sample_ids:
    graph = td.graph.IndexedRXGraph.from_geff(str(PREDICT_DIR / f"{sample_id}.geff"))[0]
    part = graph_to_rows(sample_id, graph)
    raw_rows.extend(part)
    n_nodes += sum(r["row_type"] == "node" for r in part)
    n_edges += sum(r["row_type"] == "edge" for r in part)
raw = pd.DataFrame(raw_rows, columns=COLUMNS)
raw["id"] = np.arange(len(raw), dtype=np.int64)
print("raw nodes", n_nodes, "raw edges", n_edges, "rows", len(raw))
display(raw.head(6))


<h2 id="5" style="color:#0B1F3A;border-bottom:3px solid #00B4A6;padding-bottom:6px;">5. Repair the forest</h2>

Question: which missing one-frame links are still physically plausible, and which short fragments are noise?


In [ ]:
GAP_MAX_UM = 10.0
GAP_REUSE_UM = 2.5
GAP_MAX_FRAC, GAP_MAX_ABS = 0.02, 1000
GAP_Z, GAP_YX, GAP_SHIFT_UM = 1, 4, 2.5
MIN_TRACK_LEN = 4
KEEP_DIVISIONS = True
KEEP_BOUNDARY = True
BOUNDARY_T = 2
VOXEL_UM = np.asarray([1.625, 0.40625, 0.40625], dtype=np.float64)


def dist_um(a, b):
    return float(np.linalg.norm((np.asarray(a, np.float64) - np.asarray(b, np.float64)) * VOXEL_UM))


def open_volume(dataset):
    root = zarr.open(str(image_dir / f"{dataset}.zarr"), mode="r")
    return root if hasattr(root, "shape") and len(getattr(root, "shape", ())) == 4 else root["0"]


In [ ]:
def refine_midpoint(frame, midpoint):
    z, y, x = np.rint(midpoint).astype(int).tolist()
    z0, z1 = max(0, z - GAP_Z), min(frame.shape[0], z + GAP_Z + 1)
    y0, y1 = max(0, y - GAP_YX), min(frame.shape[1], y + GAP_YX + 1)
    x0, x1 = max(0, x - GAP_YX), min(frame.shape[2], x + GAP_YX + 1)
    patch = np.asarray(frame[z0:z1, y0:y1, x0:x1], dtype=np.float64)
    if patch.size == 0:
        return midpoint
    weights = np.maximum(patch - np.percentile(patch, 20), 0)
    total = float(weights.sum())
    if not np.isfinite(total) or total <= 0:
        return midpoint
    zz = np.arange(z0, z1, dtype=np.float64)[:, None, None]
    yy = np.arange(y0, y1, dtype=np.float64)[None, :, None]
    xx = np.arange(x0, x1, dtype=np.float64)[None, None, :]
    refined = np.array([
        float((weights * zz).sum() / total),
        float((weights * yy).sum() / total),
        float((weights * xx).sum() / total),
    ])
    return midpoint if dist_um(midpoint, refined) > GAP_SHIFT_UM else refined


In [ ]:
def degrees(node_ids, edges):
    inn = {int(i): 0 for i in node_ids}
    out = {int(i): 0 for i in node_ids}
    for src, tgt in edges:
        if src in out:
            out[src] += 1
        if tgt in inn:
            inn[tgt] += 1
    return inn, out


def components(node_ids, edges):
    parent = {int(i): int(i) for i in node_ids}

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    for src, tgt in edges:
        a, b = find(int(src)), find(int(tgt))
        if a != b:
            parent[a] = b
    groups = {}
    for node in parent:
        groups.setdefault(find(node), []).append(node)
    return list(groups.values())


In [ ]:
def parse_group(group):
    nodes, edges, by_t = {}, set(), {}
    for row in group.itertuples(index=False):
        if row.row_type == "node":
            nid = int(row.node_id)
            nodes[nid] = {"t": int(row.t), "z": float(row.z), "y": float(row.y), "x": float(row.x)}
            by_t.setdefault(int(row.t), []).append(nid)
        else:
            edges.add((int(row.source_id), int(row.target_id)))
    for src, tgt in edges:
        if src not in nodes or tgt not in nodes:
            raise ValueError("dangling edge")
    return nodes, edges, by_t


In [ ]:
def gap_candidates(nodes, by_t, inn, out, t, T):
    if t >= T - 2:
        return []
    srcs = [i for i in by_t.get(t, []) if out.get(i, 0) == 0]
    tgts = [i for i in by_t.get(t + 2, []) if inn.get(i, 0) == 0]
    if not srcs or not tgts:
        return []
    a = np.array([[nodes[i]["z"], nodes[i]["y"], nodes[i]["x"]] for i in srcs], np.float64)
    b = np.array([[nodes[i]["z"], nodes[i]["y"], nodes[i]["x"]] for i in tgts], np.float64)
    cost = cdist(a * VOXEL_UM, b * VOXEL_UM)
    hung = cost.copy()
    hung[hung > GAP_MAX_UM] = 1e6
    pairs = []
    for r, c in zip(*linear_sum_assignment(hung)):
        if cost[r, c] <= GAP_MAX_UM:
            pairs.append((float(cost[r, c]), srcs[r], tgts[c]))
    pairs.sort()
    return pairs


In [ ]:
def pick_middle(nodes, by_t, inn, out, used, midpoint, t_mid, nxt):
    best_id, best_d = None, np.inf
    for mid in by_t.get(t_mid, []):
        if mid in used or inn.get(mid, 0) or out.get(mid, 0):
            continue
        pos = np.array([nodes[mid]["z"], nodes[mid]["y"], nodes[mid]["x"]], np.float64)
        d = dist_um(midpoint, pos)
        if d < best_d:
            best_id, best_d = mid, d
    if best_id is not None and best_d <= GAP_REUSE_UM:
        return best_id, True, nxt
    nid = nxt
    nodes[nid] = {"t": t_mid, "z": float(midpoint[0]), "y": float(midpoint[1]), "x": float(midpoint[2])}
    by_t.setdefault(t_mid, []).append(nid)
    inn[nid] = out[nid] = 0
    return nid, False, nxt + 1


In [ ]:
def close_gaps(nodes, edges, by_t, image, T):
    inn, out = degrees(nodes, edges)
    nxt = (max(nodes) + 1) if nodes else 0
    budget = min(GAP_MAX_ABS, max(1, int(len(nodes) * GAP_MAX_FRAC)))
    added = reused = synth = 0
    for t in range(T - 2):
        if added >= budget:
            break
        pairs = gap_candidates(nodes, by_t, inn, out, t, T)
        if not pairs:
            continue
        frame = np.asarray(image[t + 1])
        used = set()
        for _, src, tgt in pairs:
            if added >= budget or out.get(src, 0) or inn.get(tgt, 0):
                continue
            a = np.array([nodes[src]["z"], nodes[src]["y"], nodes[src]["x"]], np.float64)
            b = np.array([nodes[tgt]["z"], nodes[tgt]["y"], nodes[tgt]["x"]], np.float64)
            mid_id, was_reuse, nxt = pick_middle(
                nodes, by_t, inn, out, used, refine_midpoint(frame, (a + b) / 2), t + 1, nxt)
            used.add(mid_id)
            edges.update({(src, mid_id), (mid_id, tgt)})
            out[src] += 1; inn[mid_id] += 1; out[mid_id] += 1; inn[tgt] += 1
            added += 1
            reused += int(was_reuse); synth += int(not was_reuse)
    return added, reused, synth


In [ ]:
def filter_short(nodes, edges, T):
    inn, out = degrees(nodes, edges)
    keep, removed = set(), 0
    for comp in components(nodes, edges):
        times = [int(nodes[i]["t"]) for i in comp]
        divide = any(out.get(i, 0) >= 2 for i in comp)
        edge = min(times) <= BOUNDARY_T or max(times) >= T - 1 - BOUNDARY_T
        if len(comp) >= MIN_TRACK_LEN or (KEEP_DIVISIONS and divide) or (KEEP_BOUNDARY and edge):
            keep.update(comp)
        else:
            removed += 1
    if not keep:
        keep, removed = set(nodes), 0
    nodes = {i: nodes[i] for i in keep}
    edges = {(s, t) for s, t in edges if s in keep and t in keep}
    return nodes, edges, removed


In [ ]:
def rows_from_graph(dataset, nodes, edges, shape):
    T, Z, Y, X = shape
    rows = []
    for nid in sorted(nodes):
        n = nodes[nid]
        rows.append({
            "id": -1, "dataset": dataset, "row_type": "node", "node_id": int(nid),
            "t": int(n["t"]),
            "z": int(np.clip(round(n["z"]), 0, Z - 1)),
            "y": int(np.clip(round(n["y"]), 0, Y - 1)),
            "x": int(np.clip(round(n["x"]), 0, X - 1)),
            "source_id": -1, "target_id": -1,
        })
    for src, tgt in sorted(edges):
        rows.append({
            "id": -1, "dataset": dataset, "row_type": "edge", "node_id": -1,
            "t": -1, "z": -1, "y": -1, "x": -1, "source_id": int(src), "target_id": int(tgt),
        })
    return pd.DataFrame(rows, columns=COLUMNS)


In [ ]:
def repair_dataset(group):
    dataset = str(group["dataset"].iloc[0])
    image = open_volume(dataset)
    shape = tuple(map(int, image.shape))
    nodes, edges, by_t = parse_group(group)
    gap, reused, synth = close_gaps(nodes, edges, by_t, image, shape[0])
    before_n, before_e = len(nodes), len(edges)
    nodes, edges, short = filter_short(nodes, edges, shape[0])
    stats = {
        "gap": gap, "reused": reused, "synth": synth, "short": short,
        "nodes": len(nodes), "edges": len(edges),
        "drop_n": before_n - len(nodes), "drop_e": before_e - len(edges),
    }
    return rows_from_graph(dataset, nodes, edges, shape), stats


In [ ]:
cleaned, clean_stats = [], []
for dataset in raw["dataset"].drop_duplicates():
    frame, stats = repair_dataset(raw[raw["dataset"] == dataset].copy())
    cleaned.append(frame)
    clean_stats.append({"dataset": dataset, **stats})
    print(dataset, stats)
clean = pd.concat(cleaned, ignore_index=True)
clean["id"] = np.arange(len(clean), dtype=np.int64)
num = [c for c in COLUMNS if c not in ("dataset", "row_type")]
clean[num] = clean[num].astype(np.int64)
print(f"{len(raw)} raw rows -> {len(clean)} cleaned rows")


In [ ]:
stats = pd.DataFrame(clean_stats)
fig = go.Figure()
fig.add_bar(name="nodes", x=stats.dataset, y=stats.nodes, marker_color=TEAL)
fig.add_bar(name="edges", x=stats.dataset, y=stats.edges, marker_color=NAVY)
fig.update_layout(barmode="group", title="Cleaned graph size by FOV",
                  xaxis_title="dataset", yaxis_title="count")
draw(fig)


<div style="background:#F6F1E7;border-left:6px solid #00B4A6;padding:14px 18px;border-radius:0 12px 12px 0;margin:12px 0;"><b>Decision.</b> Close only one-frame holes within 10 µm. Reuse an isolated detection on the missing frame if it sits inside 2.5 µm; otherwise drop a brightness-refined midpoint. Then delete fragments shorter than 4 unless they divide or touch the video edge.</div>


<h2 id="6" style="color:#0B1F3A;border-bottom:3px solid #00B4A6;padding-bottom:6px;">6. Write submission.csv</h2>

Question: does every test FOV have valid node and edge rows?

In [ ]:
from hashlib import sha256

OUT_PATH = Path('/kaggle/working/submission.csv')
clean.to_csv(OUT_PATH, index=False)
check = pd.read_csv(OUT_PATH)
assert list(check.columns) == COLUMNS
assert set(check.dataset.unique()) == set(sample_ids)
assert check.id.tolist() == list(range(len(check)))
assert set(check.row_type.unique()) == {'node', 'edge'}
for dataset, group in check.groupby('dataset', sort=False):
    nodes = group.loc[group.row_type == 'node', 'node_id']
    edges = group[group.row_type == 'edge']
    assert nodes.is_unique, f'{dataset}: duplicate node IDs'
    assert edges.source_id.isin(nodes).all(), f'{dataset}: dangling source'
    assert edges.target_id.isin(nodes).all(), f'{dataset}: dangling target'
    assert not group[group.row_type == 'node'][['t', 'z', 'y', 'x']].lt(0).any().any()
print(check.groupby(['dataset', 'row_type']).size().unstack(fill_value=0))
print('submission:', OUT_PATH, 'rows:', len(check))
print('SHA-256:', sha256(OUT_PATH.read_bytes()).hexdigest())
display(check.head(8))

## Reproduce and limits

This is a **submission notebook**, with no training step. It uses the pinned support-pack model and the same no-hub inference recipe that produced the published 0.885 parent. The repaired CSV stays in `/kaggle/working/submission.csv` for Kaggle's submission picker. The local notebook has not been executed against hidden test volumes.

The next useful test is paired grouped validation across both embryos before using any fine-tuned weight. In `exp002`, a one-FOV fine-tune was worse; `exp003` is pending analysis. Do not read the 0.885 public LB as a private-LB guarantee.

Sources: [competition](https://www.kaggle.com/competitions/biohub-cell-tracking-during-development), [patched metric](https://github.com/royerlab/kaggle-cell-tracking-competition/blob/main/metrics.md), [published no-hub parent](https://www.kaggle.com/code/denpugovkin/biohub-0-885-without-the-fake-lineage-hub?scriptVersionId=350247150). Support-pack weights: `pilkwang/biohub-tracking-support-pack-50ep-v1` v10. Negative result: `exp002` in this repository; 0.973699 vs 0.978300, single-FOV diagnostic.
